# ЛР №1, раздел 4. Компьютерное моделирование: бросок треугольного футляра

**Объект:** футляр для очков — треугольная призма, боковые грани помечены 1, 2, 3.
**Предположение:** грань 1 немного тяжелее остальных, поэтому «1» будет выпадать чаще.
**Реальный эксперимент (N = 100):** 1 → 42, 2 → 31, 3 → 27.

Здесь футляр моделируется как **твёрдое тело** (6 степеней свободы): полёт под действием силы тяжести
со свободным вращением, удары о пол с отскоком и трением, остановка.
Исход каждого компьютерного броска определяется **только физикой** — никакие вероятности заранее не
задаются; частоты считаются по фактически выпавшим граням, как и в реальном эксперименте.

Исход (как в физическом эксперименте) — **грань первого касания**: боковая грань, внешняя нормаль которой
в момент первого касания пола направлена ближе всего вниз. Дополнительно записывается грань, на которой
футляр остановился.

Как пользоваться: «Среда выполнения → Выполнить всё». Параметры футляра и броска можно поменять в разделе 2.


In [ ]:
# numba ускоряет расчёт примерно в 100 раз (в Colab обычно уже установлена)
import subprocess, sys
try:
    import numba
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"])
QUICK = False   # True — быстрый прогон (N до 10^4, без перебора масс)

## 1. Код модели
Три модуля записываются в файлы рядом с ноутбуком и импортируются ниже. Физика — в `case_sim.py`, статистика и графики — в `experiment.py`, анимация — в `animate.py`.

In [ ]:
%%writefile case_sim.py
# -*- coding: utf-8 -*-
"""
Физическая модель подбрасывания треугольного футляра для очков
(ЛР №1, раздел 4 «Компьютерное моделирование»).

Объект
------
Футляр — прямая призма с равносторонним треугольником в сечении, тонкостенная
оболочка. Боковые грани пронумерованы 1, 2, 3. К грани 1 добавлена небольшая
дополнительная масса (по предположению, грань 1 «чуть тяжелее»), поэтому центр
масс смещён к грани 1, а тензор инерции считается из реального распределения масс.

Движение (твёрдое тело, 6 степеней свободы)
------------------------------------------
* полёт: центр масс движется под действием силы тяжести, вращение свободное —
  момент импульса L сохраняется, угловая скорость w = I(t)^-1 L
  (это и есть уравнения Эйлера для несимметричного волчка);
* удары о пол: импульсный метод (sequential impulses) в вершинах призмы —
  коэффициент восстановления + кулоновское трение;
* после первого касания футляр подпрыгивает, кувыркается и останавливается.

Исход испытания
---------------
Как и в физическом эксперименте — грань ПЕРВОГО касания: номер боковой грани,
внешняя нормаль которой в момент первого касания пола направлена ближе всего
вниз («этой гранью футляр ударился о пол»). Дополнительно записывается грань,
на которой футляр в итоге остановился.

Никакие вероятности в модель заранее НЕ закладываются: исход каждого броска
определяется только начальными условиями и уравнениями движения, а частоты
считаются по фактически выпавшим исходам.
"""
from dataclasses import dataclass, field

import numpy as np

try:
    from numba import njit, prange
    HAVE_NUMBA = True
except ImportError:  # без numba всё работает так же, но в ~100 раз медленнее
    HAVE_NUMBA = False
    prange = range

    def njit(*args, **kwargs):
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return lambda f: f


FACES = (1, 2, 3)
END_CAP = 0  # код исхода «остановился стоя на торце» (для длинного футляра практически не встречается)


# ----------------------------------------------------------------------------
# Параметры модели
# ----------------------------------------------------------------------------
@dataclass
class CaseParams:
    """Геометрия и массы футляра (подставьте свои измерения)."""
    side: float = 0.055        # сторона треугольного сечения, м
    length: float = 0.160      # длина футляра, м
    mass_shell: float = 0.050  # масса однородного корпуса (все 5 граней), кг
    extra_mass: float = 0.008  # дополнительная масса, равномерно распределённая по грани 1, кг


@dataclass
class TossParams:
    """Процедура броска: распределения случайных начальных условий."""
    h0: tuple = (0.9, 1.1)      # высота центра масс в момент броска, м            ~ U(a, b)
    vz: tuple = (0.5, 2.5)      # начальная вертикальная скорость (вверх), м/с      ~ U(a, b)
    v_xy: float = 0.3           # горизонтальные компоненты скорости, м/с          ~ N(0, σ)
    omega: tuple = (5.0, 25.0)  # модуль угловой скорости, рад/с                   ~ U(a, b)
    # Направление оси вращения — равномерно на сфере.
    # Начальная ориентация — равномерно распределённый случайный поворот (равномерно на SO(3)).


@dataclass
class SurfaceParams:
    """Свойства пола."""
    restitution: float = 0.30  # коэффициент восстановления (0 — абсолютно неупругий удар, 1 — упругий)
    friction: float = 0.40     # коэффициент трения скольжения


@dataclass
class SimParams:
    """Численные параметры."""
    dt: float = 1e-3      # шаг интегрирования, с
    t_max: float = 6.0    # максимальное время моделирования одного броска, с
    n_iter: int = 12      # итераций решателя контактов на шаг
    g: float = 9.81       # ускорение свободного падения, м/с²


# ----------------------------------------------------------------------------
# Геометрия и распределение масс
# ----------------------------------------------------------------------------
@dataclass
class Body:
    verts: np.ndarray        # (6, 3) вершины в связанной системе, относительно центра масс
    normals: np.ndarray      # (3, 3) внешние нормали боковых граней 1, 2, 3
    face_polys: list         # индексы вершин каждой грани (для отрисовки): 3 боковые + 2 торца
    mass: float
    inertia: np.ndarray      # (3, 3) тензор инерции относительно центра масс
    inertia_inv: np.ndarray
    com_shift: np.ndarray    # смещение центра масс от геометрического центра, м
    params: CaseParams = field(repr=False, default=None)


def _sample_rect(p0, p1, p3, n):
    """Равномерная сетка точек на параллелограмме p0, p1, (p1+p3-p0), p3."""
    s = (np.arange(n) + 0.5) / n
    u, w = np.meshgrid(s, s, indexing="ij")
    return p0 + u.reshape(-1, 1) * (p1 - p0) + w.reshape(-1, 1) * (p3 - p0)


def _sample_triangle(a, b, c, n):
    """Равномерная сетка точек на треугольнике."""
    pts = []
    for i in range(n):
        for j in range(n - i):
            u, w = (i + 1 / 3) / n, (j + 1 / 3) / n
            pts.append(a + u * (b - a) + w * (c - a))
            if i + j < n - 1:
                u2, w2 = (i + 2 / 3) / n, (j + 2 / 3) / n
                pts.append(a + u2 * (b - a) + w2 * (c - a))
    return np.array(pts)


def build_body(p: CaseParams = None) -> Body:
    """
    Связанная система координат: ось x — вдоль футляра, сечение — в плоскости yz.
    Грань 1 — «нижняя» в связанной системе (нормаль -z), грани 2 и 3 — наклонные.
    """
    p = p or CaseParams()
    a, Lc = p.side, p.length
    r_in = a / (2 * np.sqrt(3))  # радиус вписанной окружности сечения
    R_c = a / np.sqrt(3)         # радиус описанной окружности сечения
    tri = np.array([[0.0, R_c], [a / 2, -r_in], [-a / 2, -r_in]])  # вершина, правая нижняя, левая нижняя

    verts = np.array([[sx, y, z] for sx in (-Lc / 2, Lc / 2) for (y, z) in tri])
    normals = np.array([
        [0.0, 0.0, -1.0],                     # грань 1 (между правой и левой нижними вершинами)
        [0.0, np.sqrt(3) / 2, 0.5],           # грань 2 (между вершиной и правой нижней)
        [0.0, -np.sqrt(3) / 2, 0.5],          # грань 3 (между вершиной и левой нижней)
    ])
    face_polys = [[1, 2, 5, 4], [0, 1, 4, 3], [0, 3, 5, 2], [0, 2, 1], [3, 4, 5]]

    # --- распределение масс: оболочка постоянной толщины + добавка на грани 1 ---
    n = 40
    rects = [_sample_rect(verts[f[0]], verts[f[1]], verts[f[3]], n) for f in face_polys[:3]]
    caps = [_sample_triangle(verts[f[0]], verts[f[1]], verts[f[2]], n // 2) for f in face_polys[3:]]
    area_rect = a * Lc
    area_cap = np.sqrt(3) / 4 * a ** 2
    sigma = p.mass_shell / (3 * area_rect + 2 * area_cap)  # поверхностная плотность корпуса

    pts, w = [], []
    for i, r in enumerate(rects):
        m_face = sigma * area_rect + (p.extra_mass if i == 0 else 0.0)
        pts.append(r)
        w.append(np.full(len(r), m_face / len(r)))
    for c in caps:
        pts.append(c)
        w.append(np.full(len(c), sigma * area_cap / len(c)))
    pts, w = np.vstack(pts), np.concatenate(w)

    mass = w.sum()
    com = (w[:, None] * pts).sum(0) / mass
    d = pts - com
    inertia = (w[:, None, None] * ((d * d).sum(1)[:, None, None] * np.eye(3)
                                   - d[:, :, None] * d[:, None, :])).sum(0)
    # из-за симметрии (плоскости x=0 и y=0) оси x, y, z — главные оси инерции
    inertia[np.abs(inertia) < 1e-12 * np.abs(inertia).max()] = 0.0
    assert np.allclose(inertia, np.diag(np.diag(inertia))), "оси связанной системы должны быть главными"

    return Body(verts=verts - com, normals=normals, face_polys=face_polys, mass=mass,
                inertia=inertia, inertia_inv=np.linalg.inv(inertia), com_shift=com, params=p)


# ----------------------------------------------------------------------------
# Ядро моделирования (компилируется numba)
# ----------------------------------------------------------------------------
@njit(cache=True)
def quat_to_mat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    R = np.empty((3, 3))
    R[0, 0] = 1 - 2 * (y * y + z * z); R[0, 1] = 2 * (x * y - w * z);     R[0, 2] = 2 * (x * z + w * y)
    R[1, 0] = 2 * (x * y + w * z);     R[1, 1] = 1 - 2 * (x * x + z * z); R[1, 2] = 2 * (y * z - w * x)
    R[2, 0] = 2 * (x * z - w * y);     R[2, 1] = 2 * (y * z + w * x);     R[2, 2] = 1 - 2 * (x * x + y * y)
    return R


@njit(cache=True)
def _mat3_vec(M, v, out):
    for i in range(3):
        out[i] = M[i, 0] * v[0] + M[i, 1] * v[1] + M[i, 2] * v[2]


@njit(cache=True)
def _world_inertia_inv(R, Ib_inv, out):
    """I_world^-1 = R · I_body^-1 · R^T."""
    tmp = np.empty((3, 3))
    for i in range(3):
        for j in range(3):
            s = 0.0
            for k in range(3):
                s += R[i, k] * Ib_inv[k, j]
            tmp[i, j] = s
    for i in range(3):
        for j in range(3):
            s = 0.0
            for k in range(3):
                s += tmp[i, k] * R[j, k]
            out[i, j] = s


@njit(cache=True)
def down_face(R, normals):
    """Номер (1..3) боковой грани, внешняя нормаль которой направлена ближе всего вниз."""
    best, best_val = 0, 1e9
    for f in range(3):
        nz = R[2, 0] * normals[f, 0] + R[2, 1] * normals[f, 1] + R[2, 2] * normals[f, 2]
        if nz < best_val:
            best_val, best = nz, f
    return best + 1


@njit(cache=True)
def _rotate_free(q, L, I_diag, axis, h):
    """
    Точный поток свободного волчка с энергией T_i = Π_i² / (2 I_i):
    поворот тела вокруг собственной главной оси i на угол Π_i h / I_i,
    где Π = R^T L — момент импульса в связанной системе. L (в мировой системе) не меняется.
    """
    R = quat_to_mat(q)
    Pi = R[0, axis] * L[0] + R[1, axis] * L[1] + R[2, axis] * L[2]
    th = Pi * h / I_diag[axis]
    c, s = np.cos(th / 2), np.sin(th / 2)
    q0, q1, q2, q3 = q[0], q[1], q[2], q[3]
    # q ← q ⊗ (cos θ/2, sin θ/2 · e_axis)  — поворот в связанной системе
    if axis == 0:
        q[0] = q0 * c - q1 * s; q[1] = q1 * c + q0 * s; q[2] = q2 * c + q3 * s; q[3] = q3 * c - q2 * s
    elif axis == 1:
        q[0] = q0 * c - q2 * s; q[1] = q1 * c - q3 * s; q[2] = q2 * c + q0 * s; q[3] = q3 * c + q1 * s
    else:
        q[0] = q0 * c - q3 * s; q[1] = q1 * c + q2 * s; q[2] = q2 * c - q1 * s; q[3] = q3 * c + q0 * s
    nq = np.sqrt(q[0] ** 2 + q[1] ** 2 + q[2] ** 2 + q[3] ** 2)
    for j in range(4):
        q[j] /= nq


@njit(cache=True)
def simulate_one(x, v, q, L, verts, normals, Ib_inv, I_diag, mass, g, e, mu, dt, max_steps,
                 n_iter, stop_at_first, rec_every, rec):
    """
    Моделирует один бросок. x, v, q, L изменяются на месте.
      x — положение центра масс, v — его скорость, q — ориентация (кватернион w,x,y,z),
      L — момент импульса относительно центра масс (мировая система).
    rec — массив (n, 8) для записи траектории (t, x, y, z, qw, qx, qy, qz) каждые
    rec_every шагов (rec_every = 0 — не записывать).
    Возвращает (грань первого касания, шаг первого касания, вершина первого касания,
                грань остановки, шаг остановки, число записанных кадров).
    """
    nv = verts.shape[0]
    r = np.empty((nv, 3))
    z = np.empty(nv)
    idx = np.empty(nv, np.int64)
    Kn = np.empty(nv); Kt1 = np.empty(nv); Kt2 = np.empty(nv); tgt = np.empty(nv)
    lam_n = np.empty(nv); lam_t1 = np.empty(nv); lam_t2 = np.empty(nv)
    Iinv = np.empty((3, 3))
    w = np.empty(3)
    tmp = np.empty(3)
    dL = np.empty(3)

    beta, slop, v_bounce = 0.2, 5e-4, 0.1   # стабилизация проникновения и порог «упругого» удара
    v_rest, w_rest, t_rest = 0.02, 0.3, 0.2  # критерий остановки: |v|, |w| малы в течение t_rest
    inv_m = 1.0 / mass

    first_face, first_step, first_vertex = -1, -1, -1
    rest_face, rest_step = -1, -1
    still = 0
    n_rec = 0
    R = quat_to_mat(q)

    for step in range(max_steps):
        R = quat_to_mat(q)
        _world_inertia_inv(R, Ib_inv, Iinv)

        if rec_every > 0 and step % rec_every == 0 and n_rec < rec.shape[0]:
            rec[n_rec, 0] = step * dt
            rec[n_rec, 1:4] = x
            rec[n_rec, 4:8] = q
            n_rec += 1

        # --- поиск вершин, касающихся пола (z < 0) ---
        nc = 0
        zmin, kmin = 1e9, -1
        for k in range(nv):
            for j in range(3):
                r[k, j] = R[j, 0] * verts[k, 0] + R[j, 1] * verts[k, 1] + R[j, 2] * verts[k, 2]
            z[k] = x[2] + r[k, 2]
            if z[k] < zmin:
                zmin, kmin = z[k], k
            if z[k] < 0.0:
                idx[nc] = k
                nc += 1

        if nc > 0 and first_step < 0:
            first_face = down_face(R, normals)
            first_step = step
            first_vertex = kmin
            if stop_at_first:
                break

        # --- сила тяжести и угловая скорость w = I^-1 L ---
        v[2] -= g * dt
        _mat3_vec(Iinv, L, w)

        # --- удары/контакт: последовательные импульсы ---
        if nc > 0:
            for c in range(nc):
                k = idx[c]
                rx, ry, rz = r[k, 0], r[k, 1], r[k, 2]
                # эффективные массы K = 1/m + (r×n)·I^-1·(r×n) для нормали и двух касательных
                tmp[0], tmp[1], tmp[2] = ry, -rx, 0.0          # r × e_z
                _mat3_vec(Iinv, tmp, dL)
                Kn[c] = inv_m + tmp[0] * dL[0] + tmp[1] * dL[1]
                tmp[0], tmp[1], tmp[2] = 0.0, rz, -ry          # r × e_x
                _mat3_vec(Iinv, tmp, dL)
                Kt1[c] = inv_m + tmp[1] * dL[1] + tmp[2] * dL[2]
                tmp[0], tmp[1], tmp[2] = -rz, 0.0, rx          # r × e_y
                _mat3_vec(Iinv, tmp, dL)
                Kt2[c] = inv_m + tmp[0] * dL[0] + tmp[2] * dL[2]
                # нормальная скорость точки до удара и желаемая скорость после
                vn0 = v[2] + w[0] * ry - w[1] * rx
                target = -e * vn0 if vn0 < -v_bounce else 0.0
                pen = -z[k] - slop
                if pen > 0.0:
                    target = max(target, beta * pen / dt)
                tgt[c] = target
                lam_n[c] = 0.0; lam_t1[c] = 0.0; lam_t2[c] = 0.0

            for it in range(n_iter):
                for c in range(nc):
                    k = idx[c]
                    rx, ry, rz = r[k, 0], r[k, 1], r[k, 2]
                    # нормальный импульс (только отталкивание: накопленный λ ≥ 0)
                    uz = v[2] + w[0] * ry - w[1] * rx
                    dl = (tgt[c] - uz) / Kn[c]
                    new = max(lam_n[c] + dl, 0.0)
                    dl = new - lam_n[c]
                    lam_n[c] = new
                    v[2] += dl * inv_m
                    dL[0], dL[1], dL[2] = ry * dl, -rx * dl, 0.0   # r × (0, 0, dl)
                    L[0] += dL[0]; L[1] += dL[1]
                    _mat3_vec(Iinv, dL, tmp)
                    w[0] += tmp[0]; w[1] += tmp[1]; w[2] += tmp[2]

                    # трение Кулона: |λ_t| ≤ μ λ_n
                    ux = v[0] + w[1] * rz - w[2] * ry
                    uy = v[1] + w[2] * rx - w[0] * rz
                    n1 = lam_t1[c] - ux / Kt1[c]
                    n2 = lam_t2[c] - uy / Kt2[c]
                    lim = mu * lam_n[c]
                    mag = np.sqrt(n1 * n1 + n2 * n2)
                    if mag > lim:
                        s = lim / mag
                        n1 *= s; n2 *= s
                    d1, d2 = n1 - lam_t1[c], n2 - lam_t2[c]
                    lam_t1[c], lam_t2[c] = n1, n2
                    v[0] += d1 * inv_m
                    v[1] += d2 * inv_m
                    dL[0], dL[1], dL[2] = -rz * d2, rz * d1, rx * d2 - ry * d1   # r × (d1, d2, 0)
                    L[0] += dL[0]; L[1] += dL[1]; L[2] += dL[2]
                    _mat3_vec(Iinv, dL, tmp)
                    w[0] += tmp[0]; w[1] += tmp[1]; w[2] += tmp[2]

        # --- остановка ---
        speed = np.sqrt(v[0] ** 2 + v[1] ** 2 + v[2] ** 2)
        wn = np.sqrt(w[0] ** 2 + w[1] ** 2 + w[2] ** 2)
        if nc > 0 and speed < v_rest and wn < w_rest:
            still += 1
        else:
            still = 0
        if first_step >= 0 and still * dt >= t_rest:
            rest_step = step
            break

        # --- интегрирование ---
        # центр масс — полунеявный Эйлер;
        # вращение — симметричное расщепление свободного волчка по главным осям
        # (оси 1,2,3,2,1): каждый подшаг — точный поворот вокруг одной главной оси,
        # поэтому энергия и |L| не «уплывают» даже за много оборотов.
        x[0] += v[0] * dt; x[1] += v[1] * dt; x[2] += v[2] * dt
        _rotate_free(q, L, I_diag, 0, 0.5 * dt)
        _rotate_free(q, L, I_diag, 1, 0.5 * dt)
        _rotate_free(q, L, I_diag, 2, dt)
        _rotate_free(q, L, I_diag, 1, 0.5 * dt)
        _rotate_free(q, L, I_diag, 0, 0.5 * dt)

    if first_step >= 0 and not stop_at_first:
        R = quat_to_mat(q)
        # стоит на торце, если ось футляра почти вертикальна
        if abs(R[2, 0]) > 0.8:
            rest_face = 0
        else:
            rest_face = down_face(R, normals)
        if rest_step < 0:
            rest_step = max_steps
    return first_face, first_step, first_vertex, rest_face, rest_step, n_rec


@njit(parallel=True, cache=True)
def _simulate_batch(X, V, Q, Lw, verts, normals, Ib_inv, I_diag, mass, g, e, mu, dt, max_steps,
                    n_iter, stop_at_first):
    N = X.shape[0]
    out = np.empty((N, 5), np.int64)
    rec = np.empty((0, 8))
    for i in prange(N):
        res = simulate_one(X[i].copy(), V[i].copy(), Q[i].copy(), Lw[i].copy(), verts, normals,
                           Ib_inv, I_diag, mass, g, e, mu, dt, max_steps, n_iter, stop_at_first, 0, rec)
        out[i, 0] = res[0]; out[i, 1] = res[1]; out[i, 2] = res[2]
        out[i, 3] = res[3]; out[i, 4] = res[4]
    return out


# ----------------------------------------------------------------------------
# Удобный интерфейс
# ----------------------------------------------------------------------------
def random_quaternions(n, rng):
    """Равномерно распределённые повороты: нормированный 4-мерный гауссов вектор."""
    q = rng.normal(size=(n, 4))
    return q / np.linalg.norm(q, axis=1, keepdims=True)


def random_unit_vectors(n, rng):
    u = rng.normal(size=(n, 3))
    return u / np.linalg.norm(u, axis=1, keepdims=True)


class CaseModel:
    """Модель «футляр + бросок + пол»."""

    def __init__(self, case=None, toss=None, surface=None, sim=None):
        self.case = case or CaseParams()
        self.toss = toss or TossParams()
        self.surface = surface or SurfaceParams()
        self.sim = sim or SimParams()
        self.body = build_body(self.case)

    # --- начальные условия ---
    def sample_initial(self, n, rng):
        t = self.toss
        X = np.zeros((n, 3))
        X[:, 2] = rng.uniform(*t.h0, n)
        V = np.column_stack([rng.normal(0, t.v_xy, n), rng.normal(0, t.v_xy, n), rng.uniform(*t.vz, n)])
        Q = random_quaternions(n, rng)
        W = random_unit_vectors(n, rng) * rng.uniform(*t.omega, n)[:, None]
        # L = I_world · w,  I_world = R · I_body · R^T
        Lw = np.empty_like(W)
        for i in range(n):
            R = quat_to_mat(Q[i])
            Lw[i] = R @ self.body.inertia @ R.T @ W[i]
        return dict(X=X, V=V, Q=Q, L=Lw, W=W)

    def _args(self):
        b, s, sim = self.body, self.surface, self.sim
        return (b.verts, b.normals, b.inertia_inv, np.diag(b.inertia).copy(), b.mass, sim.g,
                s.restitution, s.friction,
                sim.dt, int(round(sim.t_max / sim.dt)), sim.n_iter)

    # --- серия бросков ---
    def run(self, n, seed=None, rng=None, until_rest=True):
        """
        n бросков. Возвращает словарь массивов:
          first — грань первого касания (1..3) — ИСХОД испытания,
          rest  — грань, на которой футляр остановился (0 — на торце; -1 если не считали),
          t_first, t_rest — времена, с.
        until_rest=False — моделировать только до первого касания (в несколько раз быстрее).
        """
        rng = rng or np.random.default_rng(seed)
        ic = self.sample_initial(n, rng)
        out = _simulate_batch(ic["X"], ic["V"], ic["Q"], ic["L"], *self._args(), not until_rest)
        dt = self.sim.dt
        return dict(first=out[:, 0], rest=out[:, 3] if until_rest else np.full(n, -1),
                    t_first=out[:, 1] * dt, t_rest=out[:, 4] * dt, first_vertex=out[:, 2], ic=ic)

    # --- траектория одного броска (для анимации) ---
    def trajectory(self, ic=None, rng=None, seed=None, fps=240, t_after_rest=0.3):
        """Записывает траекторию одного броска с частотой fps кадров в секунду."""
        if ic is None:
            rng = rng or np.random.default_rng(seed)
            ic = {k: v[0] for k, v in self.sample_initial(1, rng).items()}
        dt = self.sim.dt
        every = max(1, int(round(1.0 / (fps * dt))))
        max_steps = int(round(self.sim.t_max / dt))
        rec = np.empty((max_steps // every + 2, 8))
        x, v, q, L = (np.array(ic[k], dtype=float) for k in ("X", "V", "Q", "L"))
        args = self._args()
        res = simulate_one(x, v, q, L, *args[:-2], max_steps, args[-1], False, every, rec)
        first_face, first_step, first_vertex, rest_face, rest_step, n_rec = res
        rec = rec[:n_rec]
        # «досчитываем» несколько кадров покоя, чтобы анимация не обрывалась
        n_hold = int(t_after_rest * fps)
        if n_hold > 0 and n_rec > 0:
            hold = np.repeat(rec[-1:], n_hold, axis=0)
            hold[:, 0] = rec[-1, 0] + np.arange(1, n_hold + 1) / fps
            rec = np.vstack([rec, hold])
        return dict(t=rec[:, 0], x=rec[:, 1:4], q=rec[:, 4:8],
                    first_face=int(first_face), t_first=first_step * dt, first_vertex=int(first_vertex),
                    rest_face=int(rest_face), t_rest=rest_step * dt, ic=ic)

    def world_vertices(self, x, q):
        """Мировые координаты 6 вершин для положения x и ориентации q."""
        R = quat_to_mat(np.asarray(q, dtype=float))
        return np.asarray(x) + self.body.verts @ R.T

    def world_normals(self, q):
        R = quat_to_mat(np.asarray(q, dtype=float))
        return self.body.normals @ R.T


In [ ]:
%%writefile experiment.py
# -*- coding: utf-8 -*-
"""
Компьютерный эксперимент (ЛР №1, разделы 4–5): серии бросков, частоты,
накопленные частоты, сравнение с моделью и с реальным экспериментом.
"""
import numpy as np
import matplotlib.pyplot as plt

from case_sim import CaseModel, CaseParams, FACES

# ---------------------------------------------------------------------------
# Данные
# ---------------------------------------------------------------------------
# Реальный эксперимент (счётчик на фото): сколько раз выпала каждая грань.
PHYS_COUNTS = {1: 42, 2: 31, 3: 27}

# Вероятности исходной модели из раздела 2 (заданы ДО эксперимента).
# По умолчанию — симметричная модель. Если в отчёте вы задали другие p_i
# (например, p1 = 0.40, p2 = p3 = 0.30) — впишите их сюда.
P_MODEL = {1: 1 / 3, 2: 1 / 3, 3: 1 / 3}

# Цвет каждой грани — один и тот же на всех графиках и в анимации.
FACE_COLORS = {1: "#2a78d6", 2: "#eb6834", 3: "#1baf7a"}
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})


# ---------------------------------------------------------------------------
# Статистика
# ---------------------------------------------------------------------------
def counts_of(outcomes):
    outcomes = np.asarray(outcomes)
    return {f: int((outcomes == f).sum()) for f in FACES}


def freqs_of(outcomes):
    n = len(outcomes)
    return {f: c / n for f, c in counts_of(outcomes).items()}


def wilson_ci(k, n, z=1.96):
    """95%-доверительный интервал Уилсона для вероятности по k успехам из n."""
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h


def chi2_test(counts, probs):
    """
    Критерий согласия χ² Пирсона: H0 — исходы имеют вероятности probs.
    Для k = 3 исходов число степеней свободы 2, и p-value = exp(-χ²/2) точно.
    """
    n = sum(counts.values())
    chi2 = sum((counts[f] - n * probs[f]) ** 2 / (n * probs[f]) for f in FACES)
    return chi2, float(np.exp(-chi2 / 2))


def cumulative_freq(outcomes):
    """Накопленная относительная частота p̂_i(n), n = 1..N, для каждой грани."""
    outcomes = np.asarray(outcomes)
    n = np.arange(1, len(outcomes) + 1)
    return {f: np.cumsum(outcomes == f) / n for f in FACES}


# ---------------------------------------------------------------------------
# Серии
# ---------------------------------------------------------------------------
def run_series(model: CaseModel, sizes=(10, 100, 1000, 10_000, 100_000), seed=2026):
    """Независимые компьютерные эксперименты объёма N для каждого N из sizes."""
    rng = np.random.default_rng(seed)
    out = {}
    for N in sizes:
        out[N] = model.run(N, rng=rng)
    return out


def table_rows(series):
    rows = []
    for N, r in series.items():
        pf, pr = freqs_of(r["first"]), freqs_of(r["rest"])
        rows.append((N, pf, pr, int((r["rest"] == 0).sum())))
    return rows


def print_table(series):
    print(f"{'N':>7} | {'первое касание: p̂1   p̂2    p̂3':>32} | {'остановка: p̂1   p̂2    p̂3':>27}")
    print("-" * 75)
    for N, pf, pr, n_end in table_rows(series):
        print(f"{N:>7} | {pf[1]:>18.4f} {pf[2]:.4f} {pf[3]:.4f} | {pr[1]:>15.4f} {pr[2]:.4f} {pr[3]:.4f}"
              + (f"   (на торце: {n_end})" if n_end else ""))


def markdown_table(series):
    lines = ["| N | p̂₁ (1-е касание) | p̂₂ | p̂₃ | p̂₁ (остановка) | p̂₂ | p̂₃ |",
             "|---:|---:|---:|---:|---:|---:|---:|"]
    for N, pf, pr, _ in table_rows(series):
        lines.append(f"| {N} | {pf[1]:.4f} | {pf[2]:.4f} | {pf[3]:.4f} | {pr[1]:.4f} | {pr[2]:.4f} | {pr[3]:.4f} |")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Графики
# ---------------------------------------------------------------------------
def plot_convergence(outcomes, title, p_model=P_MODEL, phys_counts=PHYS_COUNTS, ax=None):
    """Накопленная частота p̂_i(n) для всех граней + p_i модели и частоты реального эксперимента."""
    cum = cumulative_freq(outcomes)
    n = np.arange(1, len(outcomes) + 1)
    n_phys = sum(phys_counts.values())
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5.2))
    for f in FACES:
        c = FACE_COLORS[f]
        ax.plot(n, cum[f], color=c, lw=2, label=f"Грань {f}: p̂{f}(n), симуляция")
        ax.axhline(phys_counts[f] / n_phys, color=c, lw=1.5, ls=":", alpha=0.9)
    # подписи финальных значений справа, раздвинутые, чтобы не налезали друг на друга
    ys = sorted((cum[f][-1], f) for f in FACES)
    pos = [ys[0][0]]
    for y, _ in ys[1:]:
        pos.append(max(y, pos[-1] + 0.045))
    for (y, f), yp in zip(ys, pos):
        ax.text(len(n) * 1.12, yp, f"p̂{f} = {y:.3f}", color=INK, va="center", fontsize=10)
    for f in FACES:
        ax.axhline(p_model[f], color=INK_2, lw=1.2, ls="--", zorder=1)
    ax.plot([], [], color=INK_2, ls="--", label="p_i исходной модели")
    ax.plot([], [], color=INK_2, ls=":", lw=1.5, label=f"реальный эксперимент (N = {n_phys}), цвет — грань")
    ax.set_xscale("log")
    ax.set_xlim(1, len(n) * 4)
    ax.set_ylim(0, 1)
    ax.set_xlabel("число компьютерных бросков n")
    ax.set_ylabel("накопленная относительная частота")
    ax.set_title(title, loc="left")
    ax.legend(loc="upper right", fontsize=9, frameon=False)
    return ax


def plot_comparison(series_big, p_model=P_MODEL, phys_counts=PHYS_COUNTS, ax=None):
    """p_i модели, p̂_i реального эксперимента (с 95% ДИ) и p̂_i симуляции (оба правила)."""
    n_phys = sum(phys_counts.values())
    groups = [
        ("Исходная\nмодель p_i", {f: p_model[f] for f in FACES}, None),
        (f"Реальный\nэксперимент\n(N = {n_phys})", {f: phys_counts[f] / n_phys for f in FACES},
         {f: wilson_ci(phys_counts[f], n_phys) for f in FACES}),
        (f"Симуляция:\nпервое касание\n(N = {len(series_big['first'])})", freqs_of(series_big["first"]), None),
        (f"Симуляция:\nгрань остановки\n(N = {len(series_big['rest'])})", freqs_of(series_big["rest"]), None),
    ]
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5.2))
    w = 0.26
    for gi, (name, p, ci) in enumerate(groups):
        for j, f in enumerate(FACES):
            x = gi + (j - 1) * (w + 0.02)
            ax.bar(x, p[f], width=w, color=FACE_COLORS[f], label=f"Грань {f}" if gi == 0 else None)
            if ci is not None:
                lo, hi = ci[f]
                ax.errorbar(x, p[f], yerr=[[p[f] - lo], [hi - p[f]]], color=INK, capsize=4, lw=1.2)
            ax.text(x, p[f] + (0.012 if ci is None else ci[f][1] - p[f] + 0.012), f"{p[f]:.2f}",
                    ha="center", va="bottom", fontsize=9, color=INK)
    ax.axhline(1 / 3, color=INK_2, lw=1, ls="--", zorder=0)
    ax.text(-0.5, 1 / 3 + 0.005, "1/3", va="bottom", ha="left", color=INK_2, fontsize=9)
    ax.set_xticks(range(len(groups)), [g[0] for g in groups])
    ax.set_xlim(-0.55, len(groups) - 0.45)
    ax.set_ylim(0, max(0.62, max(max(g[1].values()) for g in groups) + 0.12))
    ax.set_ylabel("вероятность / относительная частота")
    ax.set_title("Сравнение: модель, реальный эксперимент и симуляция", loc="left")
    ax.legend(loc="upper left", frameon=False, ncol=3)
    ax.grid(axis="x", visible=False)
    return ax


def mass_sweep(base: CaseModel, extra_masses_g=(0, 5, 10, 15, 20, 30, 40), n=20_000, seed=7):
    """Как зависит распределение исходов от «лишней» массы на грани 1."""
    res = {"dm": np.array(extra_masses_g, float), "first": [], "rest": [], "shift_mm": []}
    for dm in extra_masses_g:
        cp = CaseParams(**{**base.case.__dict__, "extra_mass": dm / 1000})
        m = CaseModel(cp, base.toss, base.surface, base.sim)
        r = m.run(n, seed=seed)
        res["first"].append(freqs_of(r["first"]))
        res["rest"].append(freqs_of(r["rest"]))
        res["shift_mm"].append(-m.body.com_shift[2] * 1000)
    return res


def plot_mass_sweep(sw, phys_counts=PHYS_COUNTS, n_sim=None):
    n_phys = sum(phys_counts.values())
    lo, hi = wilson_ci(phys_counts[1], n_phys)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, key, title in ((axes[0], "first", "Исход = грань первого касания"),
                           (axes[1], "rest", "Исход = грань, на которой футляр остановился")):
        ax.axhspan(lo, hi, color=FACE_COLORS[1], alpha=0.10, lw=0)
        ax.axhline(phys_counts[1] / n_phys, color=FACE_COLORS[1], ls=":", lw=1.5)
        for f in FACES:
            y = [p[f] for p in sw[key]]
            ax.plot(sw["dm"], y, "-o", color=FACE_COLORS[f], lw=2, ms=6,
                    mec=SURFACE, mew=1.5, label=f"Грань {f}")
        ax.set_title(title, loc="left", fontsize=12)
        ax.set_xlabel("дополнительная масса на грани 1, г")
        ax.set_ylim(0.15, 0.6)
    axes[0].set_ylabel("относительная частота")
    axes[0].text(sw["dm"][0], hi + 0.01, f"реальный эксперимент: p̂₁ = {phys_counts[1] / n_phys:.2f} (95% ДИ)",
                 color=INK_2, fontsize=9)
    axes[1].legend(loc="upper left", frameon=False)
    fig.suptitle("Влияние «тяжёлой» грани на исход" + (f" (N = {n_sim} бросков на точку)" if n_sim else ""),
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    return fig


In [ ]:
%%writefile animate.py
# -*- coding: utf-8 -*-
"""
Анимация бросков треугольного футляра (matplotlib, 3D).

  * animate_throw  — один бросок подробно: 3D-вид с камерой, следящей за футляром,
                     график высоты и график «какая грань смотрит вниз»;
                     момент первого касания замедлен и выделен.
  * animate_series — несколько бросков подряд со «счётчиком», как в реальном
                     эксперименте: частоты считаются по фактически выпавшим граням.

fx=True — «боевой» режим: неоновая сцена, лазерное сопровождение, прицел на точке
падения, взрыв с искрами и ударными волнами в момент первого касания, тряска камеры,
огненный след и HUD. Это только визуальные эффекты: физика и исходы те же самые.
fx=False — спокойная светлая версия (удобно для отчёта).
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

from case_sim import CaseModel, FACES, quat_to_mat
from experiment import FACE_COLORS, INK, INK_2, GRID, SURFACE, PHYS_COUNTS

THEMES = {
    "light": dict(bg=SURFACE, panel=SURFACE, floor="#efede7", floor_line="#d9d6cd", ink=INK, ink2=INK_2,
                  grid=GRID, edge="#2b2b2b", edge_w=0.9, cap="#a19f97", faces=FACE_COLORS, shadow_alpha=0.28),
    # тёмная тема: цвета граней — те же оттенки, подобранные для тёмного фона
    "neon": dict(bg="#07080f", panel="#0d0f1c", floor="#130a28", floor_line="#ff2bd6", ink="#f4f4f8",
                 ink2="#a9abc3", grid="#262a40", edge="#9ff6ff", edge_w=1.2, cap="#5b5f7a",
                 faces={1: "#3987e5", 2: "#d95926", 3: "#199e70"}, shadow_alpha=0.55),
}
LASER_COLORS = ("#ff3b3b", "#39ff14", "#00e5ff")
HOT = np.array([[1.0, 0.95, 0.69], [1.0, 0.82, 0.25], [1.0, 0.55, 0.10], [1.0, 0.24, 0.10]])
BOOM_WORDS = ("БАБАХ!", "БУМ!", "БДЫЩ!", "КРАШ!", "ТЫДЫЩ!")


def _rgb(hex_color):
    h = hex_color.lstrip("#")
    return np.array([int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)])


def _hull_2d(pts):
    """Выпуклая оболочка точек на плоскости (монотонная цепь Эндрю)."""
    pts = sorted(map(tuple, pts))
    if len(pts) < 3:
        return np.array(pts)

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower, upper = [], []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    return np.array(lower[:-1] + upper[:-1])


def _circle(c, r, z=2e-4, n=48):
    a = np.linspace(0, 2 * np.pi, n)
    return np.column_stack([c[0] + r * np.cos(a), c[1] + r * np.sin(a), np.full(n, z)])


# ---------------------------------------------------------------------------
# Спецэффекты
# ---------------------------------------------------------------------------
class Fx:
    """Лазеры, прицел, взрыв, ударные волны, вспышка, огненный след, HUD, звёзды."""

    def __init__(self, fig, ax, th, seed=0):
        self.fig, self.ax, self.th = fig, ax, th
        self.rng = np.random.default_rng(seed)
        glow = ((10, 0.08), (5, 0.22), (1.6, 0.95))
        self.beams = [[ax.plot([], [], [], color=c, lw=w, alpha=a, zorder=4, solid_capstyle="round")[0]
                       for w, a in glow] for c in LASER_COLORS]
        self.emit, = ax.plot([], [], [], "o", ms=6, color="#ffffff", mec="#ff2bd6", mew=1.5, ls="none", zorder=4)
        self.reticle = Line3DCollection([np.zeros((2, 3))], colors="#ff3b3b", linewidths=1.6, zorder=3)
        self.rings = Line3DCollection([np.zeros((2, 3))], linewidths=2.2, zorder=3)
        ax.add_collection3d(self.reticle)
        ax.add_collection3d(self.rings)
        self.comet = ax.scatter([0], [0], [0], s=[0], depthshade=False, zorder=4, linewidths=0)
        self.fire = ax.scatter([0], [0], [0], s=[0], depthshade=False, zorder=6, linewidths=0)
        self.sparks = ax.scatter([0], [0], [0], s=[0], depthshade=False, zorder=7, linewidths=0)
        self.boom = ax.text2D(0.5, 0.80, "", transform=ax.transAxes, ha="center", va="center", color="#ffe14d",
                              fontweight="bold", rotation=-7, zorder=10)
        self.boom.set_path_effects([pe.withStroke(linewidth=8, foreground="#ff2d2d"), pe.Normal()])
        self.hud = ax.text2D(0.02, 0.03, "", transform=ax.transAxes, ha="left", va="bottom", color="#39ff14",
                             family="monospace", fontsize=10, zorder=10)
        self.glow = ax.scatter([0], [0], [0], s=[0], depthshade=False, zorder=7, linewidths=0)
        pos = ax.get_position()
        self.flash = Rectangle((pos.x0, pos.y0), pos.width, pos.height, transform=fig.transFigure,
                               color="#fff2b0", alpha=0.0, zorder=20)
        fig.add_artist(self.flash)
        # звёздное небо — на отдельных осях под 3D-сценой
        self.sky = fig.add_axes(pos, zorder=-1)
        self.sky.set_axis_off()
        self.sky.set_xlim(0, 1); self.sky.set_ylim(0, 1)
        n = 140
        self.star_xy = self.rng.random((n, 2))
        self.star_s = self.rng.uniform(0.5, 6, n)
        self.star_ph = self.rng.uniform(0, 2 * np.pi, n)
        self.stars = self.sky.scatter(self.star_xy[:, 0], self.star_xy[:, 1], s=self.star_s, c="#cfd6ff",
                                      alpha=0.8, linewidths=0)
        ax.set_facecolor((0, 0, 0, 0))
        self.new_throw(None)

    def new_throw(self, target):
        """Новый бросок: точка падения (для прицела и взрыва) и случайные частицы взрыва."""
        rng = self.rng
        self.target = None if target is None else np.asarray(target, float)
        self.word = BOOM_WORDS[rng.integers(len(BOOM_WORDS))]
        n = 170
        az = rng.uniform(0, 2 * np.pi, n)
        el = rng.uniform(np.radians(15), np.radians(80), n)
        sp = rng.uniform(0.15, 0.75, n)
        self.sp_v = np.column_stack([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)]) * sp[:, None]
        self.sp_life = rng.uniform(0.5, 1.4, n)
        self.sp_col = HOT[rng.integers(0, len(HOT), n)]
        nf = 55
        d = rng.normal(size=(nf, 3))
        d[:, 2] = np.abs(d[:, 2]) * 1.4
        self.fi_v = d / np.linalg.norm(d, axis=1, keepdims=True) * rng.uniform(0.02, 0.12, nf)[:, None]
        self.fi_s = rng.uniform(0.6, 1.4, nf)

    def shake(self, age):
        """Тряска камеры после удара: (Δelev, Δazim) в градусах."""
        if age is None or age < 0 or age > 0.6:
            return 0.0, 0.0
        amp = 4.0 * np.exp(-age / 0.15)
        return amp * np.sin(age * 97.0), amp * np.cos(age * 73.0)

    def update(self, frame_idx, x, H, flight, age, trail, hud_lines, face_color):
        ax = self.ax
        # звёзды мерцают
        self.stars.set_sizes(self.star_s * (0.65 + 0.35 * np.sin(self.star_ph + 0.35 * frame_idx)))

        # --- лазерное сопровождение и прицел (пока футляр в полёте) ---
        tgt = self.target
        if flight and tgt is not None:
            for k, layers in enumerate(self.beams):
                a = np.radians(90 + 120 * k + 0.8 * frame_idx)
                e = tgt + np.array([0.24 * np.cos(a), 0.24 * np.sin(a), 0.0])
                for ln in layers:
                    ln.set_data_3d([e[0], x[0]], [e[1], x[1]], [0.002, x[2]])
            em = np.array([tgt + [0.24 * np.cos(np.radians(90 + 120 * k + 0.8 * frame_idx)),
                                  0.24 * np.sin(np.radians(90 + 120 * k + 0.8 * frame_idx)), 0.002] for k in range(3)])
            self.emit.set_data_3d(em[:, 0], em[:, 1], em[:, 2])
            r = 0.05 * (1 + 0.15 * np.sin(frame_idx * 0.5))
            rot = np.radians(3 * frame_idx)
            segs = [_circle(tgt, r), _circle(tgt, 0.45 * r)]
            for k in range(4):
                a = rot + k * np.pi / 2
                d = np.array([np.cos(a), np.sin(a), 0])
                segs.append(np.array([tgt + 0.6 * r * d, tgt + 1.5 * r * d]) + [0, 0, 2e-4])
            self.reticle.set_segments(segs)
        else:
            for layers in self.beams:
                for ln in layers:
                    ln.set_data_3d([], [], [])
            self.emit.set_data_3d([], [], [])
            self.reticle.set_segments([])

        # --- огненный след центра масс ---
        if trail is not None and len(trail) > 1 and flight:
            m = len(trail)
            u = np.linspace(0, 1, m)
            col = np.column_stack([np.ones(m), 0.25 + 0.7 * u, 0.1 + 0.5 * u ** 3, 0.1 + 0.8 * u])
            self.comet._offsets3d = (trail[:, 0], trail[:, 1], trail[:, 2])
            self.comet.set_sizes(4 + 60 * u ** 2)
            self.comet.set_facecolors(col)
        else:
            self.comet._offsets3d = ([], [], [])

        # --- взрыв ---
        if age is not None and age >= 0 and tgt is not None:
            c = tgt
            # искры: баллистика с «мультяшной» гравитацией, отскок от пола
            t = np.minimum(age, self.sp_life)
            p = c + self.sp_v * t[:, None] + np.array([0, 0, -0.6]) * (t[:, None] ** 2) / 2
            p[:, 2] = np.abs(p[:, 2])
            alive = age < self.sp_life
            fade = np.clip(1 - age / self.sp_life, 0, 1)
            col = np.column_stack([self.sp_col, fade])
            self.sparks._offsets3d = (p[alive, 0], p[alive, 1], p[alive, 2])
            self.sparks.set_sizes((4 + 26 * fade[alive]) * 1.0)
            self.sparks.set_facecolors(col[alive])
            # огненный шар → дым (полупрозрачный, чтобы футляр было видно)
            if age < 1.4:
                q = c + self.fi_v * (age ** 0.6) + np.array([0, 0, 0.05 * age])
                s = self.fi_s * (150 + 1600 * age ** 0.7) * (0.42 / H) ** 2
                w = min(1.0, age / 0.4)
                base = (1 - w) * np.array([1.0, 0.72, 0.18]) + w * np.array([0.30, 0.28, 0.36])
                base = 0.8 * base + 0.2 * _rgb(face_color)
                a = (0.65 - 0.35 * w) * max(0.0, 1 - age / 1.4)
                self.fire._offsets3d = (q[:, 0], q[:, 1], q[:, 2])
                self.fire.set_sizes(s)
                self.fire.set_facecolors(np.tile(np.append(base, a), (len(q), 1)))
            else:
                self.fire._offsets3d = ([], [], [])
            # светящийся шар в точке удара
            if age < 0.8:
                k = np.exp(-age / 0.25)
                self.glow._offsets3d = ([c[0]] * 3, [c[1]] * 3, [c[2] + 0.005] * 3)
                self.glow.set_sizes(np.array([9000, 3500, 900]) * (0.5 + 0.5 * k) * (0.42 / H) ** 2)
                self.glow.set_facecolors([[1, 0.85, 0.4, 0.12 * k], [1, 0.9, 0.6, 0.3 * k], [1, 1, 0.92, 0.9 * k]])
            else:
                self.glow._offsets3d = ([], [], [])
            # ударные волны по полу
            segs, cols = [], []
            for delay, color in ((0.0, "#ffe14d"), (0.10, "#ff5a1f"), (0.22, "#00e5ff")):
                a_ = age - delay
                if 0 <= a_ < 0.9:
                    segs.append(_circle(c, 0.02 + 0.32 * a_ ** 0.7))
                    cols.append(np.append(_rgb(color), max(0.0, 1 - a_ / 0.9)))
            self.rings.set_segments(segs if segs else [])
            if cols:
                self.rings.set_color(cols)
            # вспышка и надпись
            self.flash.set_alpha(float(0.22 * np.exp(-age / 0.06)))
            pop = min(1.0, age / 0.10)
            size = 14 + 50 * pop * (1.15 - 0.15 * min(1.0, age / 0.6))
            self.boom.set_text(self.word if age < 1.3 else "")
            self.boom.set_fontsize(size)
            self.boom.set_alpha(float(np.clip(1.6 - age / 0.8, 0, 1)))
        else:
            self.sparks._offsets3d = ([], [], [])
            self.fire._offsets3d = ([], [], [])
            self.glow._offsets3d = ([], [], [])
            self.rings.set_segments([])
            self.flash.set_alpha(0.0)
            self.boom.set_text("")

        self.hud.set_text("\n".join(hud_lines))


# ---------------------------------------------------------------------------
# 3D-сцена: пол, тень, футляр с цифрами на гранях, след центра масс
# ---------------------------------------------------------------------------
class Scene3D:
    def __init__(self, ax, model: CaseModel, base_size=0.42, elev=22, azim=-58, spin=0.12, zoom=1.35,
                 theme="light", fx=False, fig=None, seed=0):
        self.ax, self.model = ax, model
        self.th = THEMES[theme]
        th = self.th
        self.base_size = base_size          # минимальный размер видимой области, м
        self.elev, self.azim0, self.spin = elev, azim, spin
        self.center = None
        ax.computed_zorder = False          # порядок: пол → тень → след → футляр → цифры
        ax.set_proj_type("persp", focal_length=1.6)
        ax.set_box_aspect((1, 1, 1), zoom=zoom)
        ax.set_axis_off()
        ax.set_facecolor(th["bg"])

        self.floor = Poly3DCollection([np.zeros((4, 3))], facecolor=th["floor"], edgecolor="none", zorder=0)
        lw, alpha = (1.0, 0.75) if fx else (0.8, 1.0)
        self.grid = Line3DCollection([np.zeros((2, 3))], colors=th["floor_line"], linewidths=lw, alpha=alpha, zorder=1)
        self.grid_glow = Line3DCollection([np.zeros((2, 3))], colors=th["floor_line"], linewidths=4,
                                          alpha=0.12 if fx else 0.0, zorder=1)
        self.shadow = Poly3DCollection([np.zeros((3, 3))], facecolor="#000000", alpha=0.18,
                                       edgecolor="none", zorder=2)
        self.ring = Line3DCollection([np.zeros((2, 3))], colors=th["ink"], linewidths=1.5, zorder=3)
        self.trail, = ax.plot([], [], [], color=th["ink2"], lw=1, ls=(0, (2, 2)), alpha=0.6, zorder=3)
        self.plumb, = ax.plot([], [], [], color=th["ink2"], lw=0.8, alpha=0.5, zorder=3)
        self.prism = Poly3DCollection([np.zeros((3, 3))] * 5, edgecolor=th["edge"], linewidth=th["edge_w"], zorder=5)
        self.contact_pt, = ax.plot([], [], [], marker="*", ms=14, color=th["ink"], mec=th["bg"], mew=1.2,
                                   ls="none", zorder=6)
        for art in (self.floor, self.grid_glow, self.grid, self.shadow, self.ring, self.prism):
            ax.add_collection3d(art)
        stroke = [pe.withStroke(linewidth=2.5, foreground="#1b1b1b")]
        self.labels = [ax.text(0, 0, 0, str(f), color="white", fontsize=14, fontweight="bold",
                               ha="center", va="center", zorder=8, path_effects=stroke) for f in FACES]
        self.h_label = ax.text2D(0.02, 0.04, "", transform=ax.transAxes, color=th["ink2"], fontsize=10)
        self.fx = Fx(fig or ax.figure, ax, th, seed=seed) if fx else None

    def reset_camera(self, target=None):
        self.center = None
        if self.fx is not None:
            self.fx.new_throw(target)

    def _view_dir(self, elev, azim):
        e, a = np.radians(elev), np.radians(azim)
        return np.array([np.cos(e) * np.cos(a), np.cos(e) * np.sin(a), np.sin(e)])

    def update(self, x, q, frame_idx=0, trail=None, contact=None, ring=None, fx_age=None, hud=None):
        model, ax, th = self.model, self.ax, self.th
        d_el, d_az = self.fx.shake(fx_age) if self.fx is not None else (0.0, 0.0)
        elev, azim = self.elev + d_el, self.azim0 + self.spin * frame_idx + d_az
        ax.view_init(elev=elev, azim=azim)
        view = self._view_dir(elev, azim)
        light = view + np.array([0.0, 0.0, 1.2])
        light /= np.linalg.norm(light)

        # --- камера: следит за футляром по горизонтали, пол всегда в кадре ---
        H = max(self.base_size, x[2] + 0.6 * self.base_size)
        target = np.array([x[0], x[1]])
        self.center = target if self.center is None else self.center + 0.25 * (target - self.center)
        cx, cy = self.center
        ax.set_xlim(cx - H / 2, cx + H / 2)
        ax.set_ylim(cy - H / 2, cy + H / 2)
        ax.set_zlim(0, H)

        # --- пол с сеткой 5 см ---
        x0, x1, y0, y1 = cx - H / 2, cx + H / 2, cy - H / 2, cy + H / 2
        self.floor.set_verts([np.array([[x0, y0, 0], [x1, y0, 0], [x1, y1, 0], [x0, y1, 0]])])
        step = 0.05 if H < 0.8 else 0.10
        segs = [[(gx, y0, 0), (gx, y1, 0)] for gx in np.arange(np.ceil(x0 / step), np.floor(x1 / step) + 1) * step]
        segs += [[(x0, gy, 0), (x1, gy, 0)] for gy in np.arange(np.ceil(y0 / step), np.floor(y1 / step) + 1) * step]
        self.grid.set_segments(segs)
        self.grid_glow.set_segments(segs)

        # --- футляр ---
        V = model.world_vertices(x, q)
        R = quat_to_mat(np.asarray(q, float))
        normals = list(model.world_normals(q)) + [-R[:, 0], R[:, 0]]
        polys, colors = [], []
        for k, idx in enumerate(model.body.face_polys):
            polys.append(V[idx])
            base = _rgb(th["faces"][k + 1]) if k < 3 else _rgb(th["cap"])
            lum = 0.55 + 0.45 * max(0.0, float(normals[k] @ light))
            colors.append(np.clip(base * lum + 0.08 * (1 - lum), 0, 1))
        self.prism.set_verts(polys)
        self.prism.set_facecolor(colors)

        fs = float(np.clip(60 * 0.16 / H, 7, 16))
        for k, lab in enumerate(self.labels):
            n = normals[k]
            c = V[model.body.face_polys[k]].mean(0) + 0.004 * n
            lab.set_position_3d(c)
            lab.set_fontsize(fs)
            lab.set_visible(bool(n @ view > 0.3))

        # --- тень (свет сверху) и отвес от центра масс к полу ---
        hull = _hull_2d(V[:, :2])
        self.shadow.set_verts([np.column_stack([hull, np.full(len(hull), 1e-4)])])
        self.shadow.set_alpha(float(np.clip(th["shadow_alpha"] - 0.12 * x[2], 0.08, th["shadow_alpha"])))
        self.plumb.set_data_3d([x[0], x[0]], [x[1], x[1]], [0, max(0.0, V[:, 2].min())])
        self.h_label.set_text("" if self.fx is not None else f"высота центра масс: {x[2]:.2f} м")

        if trail is not None:   # след центра масс — только внутри видимой области
            trail = trail[(trail[:, 2] <= H) & (np.abs(trail[:, 0] - cx) <= H / 2) & (np.abs(trail[:, 1] - cy) <= H / 2)]
        if trail is not None and len(trail) > 1 and self.fx is None:
            self.trail.set_data_3d(trail[:, 0], trail[:, 1], trail[:, 2])
        else:
            self.trail.set_data_3d([], [], [])
        if contact is not None:
            self.contact_pt.set_data_3d([contact[0]], [contact[1]], [max(contact[2], 0.0)])
        else:
            self.contact_pt.set_data_3d([], [], [])
        if ring is not None and self.fx is None:   # «волна» от удара: (центр, радиус, прозрачность)
            (rx, ry), rad, alpha = ring
            self.ring.set_segments([_circle((rx, ry), rad)])
            self.ring.set_alpha(alpha)
        else:
            self.ring.set_segments([])

        if self.fx is not None:
            flight = contact is None
            face = hud.get("face", 1) if hud else 1
            self.fx.update(frame_idx, x, H, flight, fx_age, trail, (hud or {}).get("lines", []),
                           th["faces"][face])


# ---------------------------------------------------------------------------
# Вспомогательное
# ---------------------------------------------------------------------------
def frame_schedule(t_end, t_first, fps=30, speed=0.35, slow=0.06, slow_window=0.05, freeze=0.8):
    """
    Список моментов модельного времени для кадров. speed — скорость воспроизведения
    (1 = реальное время), около первого касания — замедление до slow,
    в сам момент касания — стоп-кадр длительностью freeze секунд.
    """
    times, t, frozen = [], 0.0, False
    while t <= t_end:
        times.append(t)
        if not frozen and t >= t_first:
            times[-1] = t_first
            times += [t_first] * int(freeze * fps)
            frozen = True
        s = slow if abs(t - t_first) < slow_window else speed
        t += s / fps
    times.append(t_end)
    return np.array(times)


def _style_2d(ax, th):
    ax.set_facecolor(th["panel"])
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(th["ink2"])
    ax.tick_params(colors=th["ink2"])
    ax.xaxis.label.set_color(th["ink"])
    ax.yaxis.label.set_color(th["ink"])
    ax.title.set_color(th["ink"])
    ax.grid(True, color=th["grid"], lw=0.8)


def _kinematics(traj):
    """Скорость центра масс и модуль угловой скорости по записанной траектории."""
    t, X, Q = traj["t"], traj["x"], traj["q"]
    dt = np.maximum(np.diff(t), 1e-9)
    v = np.linalg.norm(np.diff(X, axis=0), axis=1) / dt
    dots = np.abs(np.sum(Q[1:] * Q[:-1], axis=1)).clip(0, 1)
    w = 2 * np.arccos(dots) / dt
    return np.append(v, 0.0), np.append(w, 0.0)


def _hud_lines(traj, i, speed, omega, stage):
    return [f"▶ {stage}",
            f"  высота   {traj['x'][i, 2]:5.2f} м",
            f"  скорость {speed[i]:5.2f} м/с",
            f"  вращение {omega[i]:5.1f} рад/с"]


# ---------------------------------------------------------------------------
# Один бросок — подробно
# ---------------------------------------------------------------------------
def animate_throw(model: CaseModel, traj=None, seed=11, fps=30, speed=0.4, slow=0.05, fx=True):
    traj = traj or model.trajectory(seed=seed, fps=1000, t_after_rest=0.4)
    th = THEMES["neon" if fx else "light"]
    t, X, Q = traj["t"], traj["x"], traj["q"]
    dt_rec = t[1] - t[0]
    tf, face = traj["t_first"], traj["first_face"]
    i_first = int(round(tf / dt_rec))
    V_first = model.world_vertices(X[i_first], Q[i_first])
    contact = V_first[traj["first_vertex"]]
    v_cm, w_mag = _kinematics(traj)

    # «насколько грань смотрит вниз»: cos угла между внешней нормалью и направлением вниз
    down = np.array([-model.world_normals(q)[:, 2] for q in Q])
    zmin = np.array([model.world_vertices(x, q)[:, 2].min() for x, q in zip(X, Q)])

    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    fig.patch.set_facecolor(th["bg"])
    gs = fig.add_gridspec(2, 2, width_ratios=[1.45, 1], left=0.01, right=0.97, top=0.86, bottom=0.08,
                          hspace=0.45, wspace=0.12)
    ax3 = fig.add_subplot(gs[:, 0], projection="3d")
    axz = fig.add_subplot(gs[0, 1])
    axd = fig.add_subplot(gs[1, 1])
    scene = Scene3D(ax3, model, theme="neon" if fx else "light", fx=fx, fig=fig, seed=seed)
    scene.reset_camera(contact)

    fig.text(0.015, 0.95, "Бросок треугольного футляра", fontsize=17, fontweight="bold", color=th["ink"])
    sub = fig.text(0.015, 0.905, "", fontsize=11, color=th["ink2"])
    banner = fig.text(0.015, 0.875, "", fontsize=14, fontweight="bold", color=th["ink"], va="top", zorder=30)

    # график высоты
    _style_2d(axz, th)
    axz.plot(t, X[:, 2], color=th["ink"], lw=2, label="центр масс")
    axz.plot(t, zmin, color=th["ink2"], lw=1.2, ls="--", label="нижняя точка футляра")
    axz.axvline(tf, color=th["ink2"], lw=1, ls=":")
    axz.text(tf, X[:, 2].max() * 1.02, " первое касание", color=th["ink2"], fontsize=9, va="bottom")
    axz.set_xlim(0, t[-1])
    axz.set_ylim(-0.02, X[:, 2].max() * 1.15)
    axz.set_xlabel("время, с")
    axz.set_ylabel("высота, м")
    axz.set_title("Высота над полом", loc="left", fontsize=12, fontweight="bold", color=th["ink"])
    axz.legend(frameon=False, fontsize=9, loc="center right", labelcolor=th["ink"])
    dot_z, = axz.plot([], [], "o", color=th["ink"], ms=7, mec=th["panel"], mew=1.5)

    # график «какая грань смотрит вниз»
    _style_2d(axd, th)
    for k, f in enumerate(FACES):
        axd.plot(t, down[:, k], color=th["faces"][f], lw=2, label=f"грань {f}")
    axd.axvline(tf, color=th["ink2"], lw=1, ls=":")
    axd.plot([tf], [down[i_first, face - 1]], "o", color=th["faces"][face], ms=9, mec=th["ink"], mew=1.2)
    axd.set_xlim(0, t[-1])
    axd.set_ylim(-1.05, 1.5)
    axd.set_xlabel("время, с")
    axd.set_ylabel("cos(нормаль, вниз)")
    axd.set_title("Какая грань смотрит вниз (1 — точно вниз)", loc="left", fontsize=12, fontweight="bold", color=th["ink"])
    axd.legend(frameon=False, fontsize=9, ncol=3, loc="upper center", labelcolor=th["ink"])
    cur_d = axd.axvline(0, color=th["ink"], lw=1)

    times = frame_schedule(t[-1], tf, fps=fps, speed=speed, slow=slow)
    freeze_frames = np.flatnonzero(np.isclose(times, tf))
    j_impact = int(freeze_frames[0])
    n_frames = len(times)
    rest_face = traj["rest_face"]

    def draw(j):
        tt = times[j]
        i = min(int(round(tt / dt_rec)), len(t) - 1)
        after = j >= j_impact
        trail = X[max(0, i - 250):i + 1:4]
        ring = None
        if after and tt - tf < 0.25:
            age = (tt - tf) / 0.25 if j > freeze_frames[-1] else (j - j_impact) / max(1, len(freeze_frames))
            ring = (contact[:2], 0.01 + 0.07 * age, 1 - age)
        stage = "ЛАЗЕРНОЕ СОПРОВОЖДЕНИЕ" if not after else "ЦЕЛЬ ПОРАЖЕНА"
        scene.update(X[i], Q[i], frame_idx=j, trail=trail, contact=contact if after else None, ring=ring,
                     fx_age=(j - j_impact) / fps if after else None,
                     hud={"lines": _hud_lines(traj, i, v_cm, w_mag, stage), "face": face})
        dot_z.set_data([t[i]], [X[i, 2]])
        cur_d.set_xdata([t[i], t[i]])
        mode = "стоп-кадр" if j in freeze_frames else (
            f"замедление ×{1 / slow:.0f}" if abs(tt - tf) < 0.05 else f"замедление ×{1 / speed:.1f}")
        sub.set_text(f"t = {t[i]:.3f} с   ·   {mode}")
        if after:
            txt = f"Первое касание: грань {face}  ← исход испытания"
            if t[i] >= traj["t_rest"] - 1e-9:
                txt += f"\nОстановился на грани {rest_face}" if rest_face > 0 else "\nОстановился на торце"
            banner.set_text(txt)
            banner.set_bbox(dict(boxstyle="round,pad=0.35", fc=th["bg"], ec=th["faces"][face], lw=2.5))
        else:
            banner.set_text("")
            banner.set_bbox(None)
        return []

    anim = FuncAnimation(fig, draw, frames=n_frames, interval=1000 / fps, blit=False)
    return fig, anim, traj


# ---------------------------------------------------------------------------
# Серия бросков со «счётчиком»
# ---------------------------------------------------------------------------
def animate_series(model: CaseModel, n_throws=10, seed=3, fps=24, speed=0.7, phys_counts=PHYS_COUNTS, fx=True):
    th = THEMES["neon" if fx else "light"]
    rng = np.random.default_rng(seed)
    trajs = [model.trajectory(rng=rng, fps=1000, t_after_rest=0.25) for _ in range(n_throws)]
    contacts, kin = [], []
    for tr in trajs:
        i_f = int(round(tr["t_first"] / (tr["t"][1] - tr["t"][0])))
        contacts.append(model.world_vertices(tr["x"][i_f], tr["q"][i_f])[tr["first_vertex"]])
        kin.append(_kinematics(tr))

    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    fig.patch.set_facecolor(th["bg"])
    gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1], left=0.01, right=0.96, top=0.84, bottom=0.12, wspace=0.12)
    ax3 = fig.add_subplot(gs[0], projection="3d")
    axb = fig.add_subplot(gs[1])
    scene = Scene3D(ax3, model, spin=0.0, azim=-62, theme="neon" if fx else "light", fx=fx, fig=fig, seed=seed)

    fig.text(0.015, 0.95, "Серия компьютерных бросков", fontsize=17, fontweight="bold", color=th["ink"])
    sub = fig.text(0.015, 0.905, "", fontsize=11, color=th["ink2"])
    banner = fig.text(0.015, 0.865, "", fontsize=14, fontweight="bold", color=th["ink"], va="top", zorder=30)

    n_phys = sum(phys_counts.values())
    _style_2d(axb, th)
    axb.grid(axis="x", visible=False)
    bars = axb.bar(FACES, [0, 0, 0], color=[th["faces"][f] for f in FACES], width=0.6)
    for f in FACES:
        p = phys_counts[f] / n_phys
        axb.plot([f - 0.38, f + 0.38], [p, p], color=th["ink"], lw=2, ls=(0, (1, 1.2)))
    axb.plot([], [], color=th["ink"], lw=2, ls=(0, (1, 1.2)), label=f"реальный эксперимент (N = {n_phys})")
    axb.set_xticks(FACES, [f"грань {f}" for f in FACES])
    axb.set_ylim(0, 1.15)
    axb.set_yticks(np.linspace(0, 1, 6))
    axb.set_ylabel("относительная частота p̂ᵢ = nᵢ / n")
    axb.set_title("Частоты по фактически выпавшим граням", loc="left", fontsize=12, fontweight="bold", color=th["ink"])
    axb.legend(frameon=False, loc="upper right", fontsize=9, labelcolor=th["ink"])
    bar_txt = [axb.text(f, 0.01, "", ha="center", va="bottom", fontsize=11, color=th["ink"]) for f in FACES]
    plus_one = axb.text(1, 0.5, "", ha="center", va="bottom", fontsize=22, fontweight="bold", color="#ffe14d",
                        path_effects=[pe.withStroke(linewidth=4, foreground="#ff2d2d")])

    # расписание кадров: (номер броска, индекс записи)
    frames = []
    for b, tr in enumerate(trajs):
        dt_rec = tr["t"][1] - tr["t"][0]
        for tt in np.arange(0, tr["t"][-1], speed / fps):
            frames.append((b, min(int(round(tt / dt_rec)), len(tr["t"]) - 1)))
        frames += [(b, len(tr["t"]) - 1)] * int(0.4 * fps)

    counts = {f: 0 for f in FACES}
    state = {"b": -1, "j_hit": None}

    def draw(j):
        b, i = frames[j]
        tr = trajs[b]
        if b != state["b"]:
            state.update(b=b, j_hit=None)
            scene.reset_camera(contacts[b])
        tt = tr["t"][i]
        after = tt >= tr["t_first"] - 1e-9
        face = tr["first_face"]
        if after and state["j_hit"] is None:
            counts[face] += 1
            state["j_hit"] = j
        age = (j - state["j_hit"]) / fps if after else None
        contact = contacts[b]
        ring = (contact[:2], 0.01 + 0.07 * (tt - tr["t_first"]) / 0.3, 1 - (tt - tr["t_first"]) / 0.3) \
            if after and tt - tr["t_first"] < 0.3 else None
        stage = "ЛАЗЕРНОЕ СОПРОВОЖДЕНИЕ" if not after else "ЦЕЛЬ ПОРАЖЕНА"
        scene.update(tr["x"][i], tr["q"][i], frame_idx=j, trail=tr["x"][max(0, i - 250):i + 1:4],
                     contact=contact if after else None, ring=ring, fx_age=age,
                     hud={"lines": _hud_lines(tr, i, *kin[b], stage), "face": face})

        n = sum(counts.values())
        for k, f in enumerate(FACES):
            p = counts[f] / n if n else 0
            bars[k].set_height(p)
            bar_txt[k].set_text(f"{counts[f]}\n{p:.2f}" if n else "0")
            bar_txt[k].set_y(p + 0.01)
            glow = fx and after and age is not None and age < 0.6 and f == face
            bars[k].set_edgecolor("#ffe14d" if glow else "none")
            bars[k].set_linewidth(3 if glow else 0)
        if fx and after and age is not None and age < 0.9:
            p = counts[face] / n
            plus_one.set_text("+1")
            plus_one.set_position((face, min(p + 0.1 + 0.12 * age, 1.02)))
            plus_one.set_alpha(float(np.clip(1.5 - age / 0.6, 0, 1)))
        else:
            plus_one.set_text("")
        sub.set_text(f"бросок {b + 1} из {len(trajs)}   ·   t = {tt:.2f} с   ·   "
                     f"счётчик: 1 → {counts[1]},  2 → {counts[2]},  3 → {counts[3]}   (n = {n})")
        if after:
            banner.set_text(f"Первое касание: грань {face}")
            banner.set_bbox(dict(boxstyle="round,pad=0.35", fc=th["bg"], ec=th["faces"][face], lw=2.5))
        else:
            banner.set_text("")
            banner.set_bbox(None)
        return []

    anim = FuncAnimation(fig, draw, frames=len(frames), interval=1000 / fps, blit=False)
    return fig, anim, trajs


# ---------------------------------------------------------------------------
def save(anim, path, fps, dpi=100):
    """Сохраняет анимацию в .mp4 (нужен ffmpeg) или .gif (Pillow)."""
    fc = anim._fig.get_facecolor()
    if str(path).endswith(".gif"):
        anim.save(path, writer=PillowWriter(fps=fps), dpi=dpi, savefig_kwargs={"facecolor": fc})
    else:
        anim.save(path, writer=FFMpegWriter(fps=fps, codec="libx264", extra_args=["-pix_fmt", "yuv420p"]),
                  dpi=dpi, savefig_kwargs={"facecolor": fc})


def mp4_to_gif(mp4_path, gif_path, fps=15, width=960):
    """Быстро делает .gif из готового .mp4 (без повторной отрисовки кадров)."""
    import subprocess
    vf = (f"fps={fps},scale={width}:-1:flags=lanczos,split[a][b];"
          "[a]palettegen=max_colors=128[p];[b][p]paletteuse=dither=bayer")
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(mp4_path), "-vf", vf, str(gif_path)], check=True)


def save_all(anim, base_path, fps):
    """Сохраняет base_path.mp4 и base_path.gif (если ffmpeg недоступен — только .gif через Pillow)."""
    import shutil
    if shutil.which("ffmpeg"):
        save(anim, base_path + ".mp4", fps)
        mp4_to_gif(base_path + ".mp4", base_path + ".gif", fps=min(fps, 15))
    else:
        save(anim, base_path + ".gif", fps=fps, dpi=80)


## 2. Параметры модели

Подставьте измерения своего футляра (сторона треугольника, длина, масса) и условия броска.
`extra_mass` — насколько грань 1 тяжелее (дополнительная масса, равномерно размазанная по грани 1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

from case_sim import CaseModel, CaseParams, TossParams, SurfaceParams, SimParams
from experiment import *
from animate import animate_throw, animate_series, save_all

case = CaseParams(side=0.055, length=0.160, mass_shell=0.050, extra_mass=0.008)
toss = TossParams(h0=(0.9, 1.1), vz=(0.5, 2.5), v_xy=0.3, omega=(5.0, 25.0))
surface = SurfaceParams(restitution=0.30, friction=0.40)
model = CaseModel(case, toss, surface, SimParams())

b = model.body
print(f"масса {b.mass*1000:.0f} г, центр масс смещён к грани 1 на {-b.com_shift[2]*1000:.2f} мм")
print("главные моменты инерции, кг·м²:", np.round(np.diag(b.inertia), 7))
print("реальный эксперимент:", PHYS_COUNTS, "  модель p_i:", P_MODEL)

## 3. Визуализация одного броска

Слева — 3D-вид (камера следит за футляром, пол всегда в кадре), справа — высота над полом и
«какая грань смотрит вниз» (косинус угла между внешней нормалью грани и направлением вниз).
Момент первого касания замедлен; исход — грань с наибольшим косинусом в этот момент.

`fx=True` — с лазерами, взрывом и прочими спецэффектами; `fx=False` — спокойная версия для отчёта.
Отрисовка занимает около минуты.

In [ ]:
fig, anim, traj = animate_throw(model, seed=11, fx=True)
plt.close(fig)
save_all(anim, "throw", fps=30)
print("грань первого касания:", traj["first_face"], "  остановился на грани:", traj["rest_face"])
display(Video("throw.mp4", embed=True, width=960))

In [ ]:
# та же анимация без спецэффектов (для отчёта)
fig, anim, _ = animate_throw(model, seed=11, fx=False)
plt.close(fig)
save_all(anim, "throw_clean", fps=30)
display(Video("throw_clean.mp4", embed=True, width=960))

### Серия бросков со «счётчиком»
Частоты обновляются по фактически выпавшим граням (пунктир — реальный эксперимент). Отрисовка — пара минут.

In [ ]:
fig, anim, trajs = animate_series(model, n_throws=6 if QUICK else 10, seed=3, fx=True)
plt.close(fig)
save_all(anim, "series", fps=24)
display(Video("series.mp4", embed=True, width=960))

## 4. Компьютерный эксперимент для N = 10, 10², 10³, 10⁴, 10⁵

In [ ]:
sizes = (10, 100, 1000, 10_000) if QUICK else (10, 100, 1000, 10_000, 100_000)
series = run_series(model, sizes)
print_table(series)
big = series[max(sizes)]

In [ ]:
plot_convergence(big["first"], f"Накопленная частота: исход = грань первого касания (N = {len(big['first'])})")
plt.show()
plot_convergence(big["rest"], f"Накопленная частота: исход = грань остановки (N = {len(big['rest'])})")
plt.show()

## 5. Сравнение: модель p_i, реальный эксперимент, симуляция

In [ ]:
plot_comparison(big)
plt.show()
n_phys = sum(PHYS_COUNTS.values())
for f in (1, 2, 3):
    lo, hi = wilson_ci(PHYS_COUNTS[f], n_phys)
    print(f"грань {f}: p̂_phys = {PHYS_COUNTS[f]/n_phys:.2f}, 95% ДИ [{lo:.3f}; {hi:.3f}]")
for name, p in [("модель p_i", P_MODEL), ("симуляция, первое касание", freqs_of(big["first"])),
                ("симуляция, грань остановки", freqs_of(big["rest"]))]:
    chi2, pv = chi2_test(PHYS_COUNTS, p)
    print(f"реальные данные vs {name}: χ² = {chi2:.2f}, p-value = {pv:.3f}")

## 6. Как влияет «тяжёлая» грань

Перебираем дополнительную массу на грани 1. В полёте сила тяжести приложена к центру масс и не создаёт
момента, поэтому ориентация к моменту первого касания от неё почти не зависит. Тяжёлая грань начинает
«работать» только после удара — когда футляр подпрыгивает, кувыркается и укладывается.

In [ ]:
if not QUICK:
    sw = mass_sweep(model, n=20_000)
    plot_mass_sweep(sw, n_sim=20_000)
    plt.show()